<a href="https://colab.research.google.com/github/munnurumahesh03-coder/kaggle-predicting-loan-payback/blob/main/05_XGBoost_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Imports and Setup (Corrected)

# --- Core Libraries ---
import os
import gc
import warnings
import json

# --- Data Handling ---
import pandas as pd
import numpy as np

# --- Machine Learning & Validation ---
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, make_scorer
from sklearn.preprocessing import LabelEncoder

# --- Hyperparameter Tuning ---
import optuna

# --- Utility ---
from tqdm.notebook import tqdm
import joblib

# --- Global Configuration & Setup ---
# Suppress warnings for a cleaner notebook
warnings.filterwarnings('ignore')

# Set pandas display options to show all columns
pd.set_option('display.max_columns', None)

# --- Project-Specific Constants ---
# Define the target column name
TARGET = 'loan_paid_back'

# Define the number of splits for cross-validation
N_SPLITS = 5

# Define the number of trials for Optuna tuning
N_TRIALS_XGB = 50

# Define a random state for reproducibility
RANDOM_STATE = 42

# --- CORRECTED FILE PATHS ---
# Define the full paths to the input files based on your notebook's data source
# This points to the output from your feature engineering notebook
TRAIN_PATH = '/kaggle/input/02-feature-engineering-ipynb/train_featured_v2.csv'
TEST_PATH = '/kaggle/input/02-feature-engineering-ipynb/test_featured_v2.csv'


print("Cell 1 executed: All libraries imported and constants set.")
print(f"XGBoost version: {xgb.__version__}")
print(f"Optuna version: {optuna.__version__}")


Cell 1 executed: All libraries imported and constants set.
XGBoost version: 2.0.3
Optuna version: 4.5.0


In [ ]:
# --- Cell 2 (v2): Load Data and Correctly Classify Binary Features ---

print("--- Cell 2 (v2): Loading and Preparing Data with Corrected Feature Types ---")

# --- Define File Paths ---
TRAIN_PATH = '/kaggle/input/02-feature-engineering-ipynb/train_featured_v2.csv'
TEST_PATH = '/kaggle/input/02-feature-engineering-ipynb/test_featured_v2.csv'

# --- Load Datasets ---
try:
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)
    print("Datasets loaded successfully.")
    print(f"Train data shape: {train_df.shape}")
    print(f"Test data shape: {test_df.shape}")
except FileNotFoundError:
    print("❌ ERROR: Data files not found. Please ensure Notebook 02's output is added as input to this notebook.")
    raise

# --- Prepare Data for Modeling ---
X = train_df.drop(columns=[TARGET])
y = train_df[TARGET]
X_test = test_df

# --- Identify Feature Types (Your Improvement) ---
# 1. Initially, separate columns by data type
categorical_features_initial = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_features_initial = X.select_dtypes(include=np.number).columns.tolist()

# 2. Find binary (0/1) columns within the numerical list
binary_features = [col for col in numerical_features_initial if X[col].nunique() == 2 and X[col].min() == 0 and X[col].max() == 1]

# 3. Create the final, correct lists
# Your plan: Move binary features from numerical to categorical
numerical_features = [col for col in numerical_features_initial if col not in binary_features]
categorical_features = categorical_features_initial + binary_features

print(f"\nFound {len(binary_features)} binary (0/1) features: {binary_features}")
print("These will be treated as CATEGORICAL features as you suggested.")

# 4. The 'id' column is an identifier, not a feature
if 'id' in numerical_features:
    numerical_features.remove('id')
if 'id' in X_test.columns:
    test_ids = X_test['id']
    X_test = X_test.drop(columns=['id'])
else:
    test_ids = pd.Series(range(len(X_test)), name='id')

# --- Verification ---
print("\nData preparation complete.")
print(f"Number of features: {len(X.columns)}")
print(f"   - True Numerical features: {len(numerical_features)}")
print(f"   - Categorical (including binary): {len(categorical_features)}")
print(f"Target variable '{TARGET}' isolated.")
print(f"Test IDs captured. Shape: {test_ids.shape}")

# --- Clean up memory ---
del train_df, test_df
gc.collect()

print("\n--- Cell 2 (v2) Complete ---")

--- Cell 2 (v2): Loading and Preparing Data with Corrected Feature Types ---
Datasets loaded successfully.
Train data shape: (593994, 24)
Test data shape: (254569, 24)

Found 5 binary (0/1) features: ['is_unemployed', 'is_student', 'is_retired', 'is_home_or_business_loan', 'is_medical_or_edu_loan']
These will be treated as CATEGORICAL features as you suggested.

Data preparation complete.
Number of features: 23
   - True Numerical features: 12
   - Categorical (including binary): 11
Target variable 'loan_paid_back' isolated.
Test IDs captured. Shape: (254569,)

--- Cell 2 (v2) Complete ---


# **Automated Hyperparameter Tuning For XGBoost**

In [ ]:
# Cell 3: Feature Preprocessing & Selection

print("--- Cell 3: Preprocessing Features and Performing Selection ---")

# --- 1. Label Encoding for Tree-Based Models ---
# We will label encode all categorical features first. This is a good format for tree models
# and a necessary step before one-hot encoding.
print("Label Encoding all categorical features...")

# Create a copy to avoid SettingWithCopyWarning
X_processed = X.copy()
X_test_processed = X_test.copy()

# Use a dictionary to store the encoders for each column
label_encoders = {}

# Loop through all categorical columns (original text + binary)
for col in tqdm(categorical_features, desc="Label Encoding"):
    le = LabelEncoder()

    # Fit on the combined data from both train and test sets to ensure all categories are captured
    combined_data = pd.concat([X_processed[col], X_test_processed[col]], axis=0).astype(str)
    le.fit(combined_data)

    # Transform train and test sets
    X_processed[col] = le.transform(X_processed[col].astype(str))
    X_test_processed[col] = le.transform(X_test_processed[col].astype(str))

    # Store the fitted encoder
    label_encoders[col] = le

print(f"Label Encoding complete for {len(categorical_features)} features.")


# --- 2. One-Hot Encoding (for XGBoost stability) ---
# As discussed, for XGBoost, it's safest to one-hot encode low-cardinality categoricals.
# We will one-hot encode the original (non-binary) categorical features.
low_cardinality_cats = [col for col in categorical_features if col not in binary_features]

print(f"\nOne-Hot Encoding {len(low_cardinality_cats)} low-cardinality features: {low_cardinality_cats}")

X_processed = pd.get_dummies(X_processed, columns=low_cardinality_cats, drop_first=True)
X_test_processed = pd.get_dummies(X_test_processed, columns=low_cardinality_cats, drop_first=True)

# Align columns after one-hot encoding to ensure train and test sets have the exact same columns
# This handles cases where a category might appear in train but not test, or vice-versa.
X_processed, X_test_processed = X_processed.align(X_test_processed, join='inner', axis=1)

print("One-Hot Encoding complete.")
print(f"Shape after OHE: Train={X_processed.shape}, Test={X_test_processed.shape}")


# --- 3. Feature Selection with SelectKBest ---
from sklearn.feature_selection import SelectKBest, f_classif

# We'll select all features. This number can be tuned as a hyperparameter itself,
# but 200 is a strong starting point.
N_FEATURES_TO_SELECT = 'all'

print(f"\nPerforming feature selection with SelectKBest to find the top {N_FEATURES_TO_SELECT} features...")

# Initialize SelectKBest. f_classif is a good choice for classification tasks.
selector = SelectKBest(score_func=f_classif, k=N_FEATURES_TO_SELECT)

# Fit on the training data and transform both train and test sets
X_final = selector.fit_transform(X_processed, y)
X_test_final = selector.transform(X_test_processed)

# Get the names of the selected columns
selected_mask = selector.get_support()
selected_features = X_processed.columns[selected_mask]

# Convert the result back to a pandas DataFrame with the correct column names
X_final = pd.DataFrame(X_final, columns=selected_features)
X_test_final = pd.DataFrame(X_test_final, columns=selected_features)

print("Feature selection complete.")
print(f"Final data shape for modeling: Train={X_final.shape}, Test={X_test_final.shape}")
print(f"\nTop 5 selected features: {selected_features.tolist()[:5]}")


# --- Final Cleanup ---
del X, X_test, X_processed, X_test_processed
gc.collect()

print("\n--- Cell 3 Complete ---")


--- Cell 3: Preprocessing Features and Performing Selection ---
Label Encoding all categorical features...


Label Encoding:   0%|          | 0/11 [00:00<?, ?it/s]

Label Encoding complete for 11 features.

One-Hot Encoding 6 low-cardinality features: ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
One-Hot Encoding complete.
Shape after OHE: Train=(593994, 66), Test=(254569, 66)

Performing feature selection with SelectKBest to find the top all features...
Feature selection complete.
Final data shape for modeling: Train=(593994, 66), Test=(254569, 66)

Top 5 selected features: ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']

--- Cell 3 Complete ---


In [ ]:
pip install optuna-integration[xgboost] -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 3.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Cell 4: The ModelTuner Class (Optimized Version)

print("--- Cell 4: Defining the OPTIMIZED ModelTuner Class ---")

# Import the specific callback for pruning
from optuna.integration import XGBoostPruningCallback

class ModelTuner:
    """
    An OPTIMIZED class for XGBoost tuning.
    - Checks for GPU once during initialization.
    - Converts data to NumPy for faster slicing in the loop.
    - Uses the efficient XGBoostPruningCallback.
    """
    def __init__(self, model_class, X, y, n_splits=N_SPLITS, random_state=RANDOM_STATE):
        self.model_class = model_class
        # --- OPTIMIZATION 1: Convert to NumPy once ---
        self.X = X.values
        self.y = y.values
        self.n_splits = n_splits
        self.random_state = random_state
        self.best_params_ = None
        self.best_score_ = -1

        # --- OPTIMIZATION 2: Check for GPU once ---
        self.gpu_params = {}
        try:
            gpu_info = os.popen('nvidia-smi -L').read()
            if 'P100' in gpu_info or 'T4' in gpu_info or 'A100' in gpu_info:
                print("✅ GPU detected. Enabling GPU acceleration with new 'device=cuda' syntax.")
                self.gpu_params = {'tree_method': 'hist', 'device': 'cuda'}

            else:
                print("ℹ️ No high-performance GPU detected. Using CPU-based 'hist'.")
                self.gpu_params = {'tree_method': 'hist'}
        except Exception as e:
            print(f"⚠️ GPU detection failed: {e}. Using CPU-based 'hist'.")
            self.gpu_params = {'tree_method': 'hist'}


    def _objective(self, trial):
        """
        The objective function that Optuna will try to maximize.
        """
        # 1. Define the hyperparameter search space
        params = {
            'objective': 'binary:logistic',
            'eval_metric': 'auc',
            'booster': 'gbtree',
            'n_estimators': 2000, # Increased for more aggressive early stopping
            'verbosity': 0,
            'use_label_encoder': False,
            'seed': self.random_state,
            **self.gpu_params, # Add the pre-checked GPU params

            # --- Refined Hyperparameter Space ---
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 12),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
            'lambda': trial.suggest_float('lambda', 1e-8, 2.0, log=True),
            'alpha': trial.suggest_float('alpha', 1e-8, 2.0, log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        }

        # 2. Perform cross-validation
        cv = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        scores = []

        # --- OPTIMIZATION 3: Use the clean XGBoostPruningCallback ---
        pruning_callback = XGBoostPruningCallback(trial, "validation_0-auc")

        for fold, (train_idx, val_idx) in enumerate(cv.split(self.X, self.y)):
            # Use faster NumPy slicing
            X_train, X_val = self.X[train_idx], self.X[val_idx]
            y_train, y_val = self.y[train_idx], self.y[val_idx]

            model = self.model_class(**params)
            model.fit(X_train, y_train,
                      eval_set=[(X_val, y_val)],
                      callbacks=[pruning_callback, xgb.callback.EarlyStopping(rounds=50, save_best=True)],
                      verbose=False)

            preds = model.predict_proba(X_val)[:, 1]
            scores.append(roc_auc_score(y_val, preds))

        return np.mean(scores)

    def tune(self, n_trials, direction='maximize'):
        """
        Run the hyperparameter tuning process.
        """
        # HyperbandPruner is still an excellent choice
        pruner = optuna.pruners.HyperbandPruner(min_resource=1, max_resource=self.n_splits, reduction_factor=3)

        study = optuna.create_study(direction=direction, pruner=pruner)
        study.optimize(self._objective, n_trials=n_trials, show_progress_bar=True)

        self.best_params_ = study.best_params
        self.best_score_ = study.best_value

        print(f"\nBest Score (ROC-AUC): {self.best_score_}")
        print("Best Parameters:")
        for key, value in self.best_params_.items():
            print(f"  {key}: {value}")

        return study

print("Cell 4 executed: OPTIMIZED ModelTuner class is now defined.")


--- Cell 4: Defining the OPTIMIZED ModelTuner Class ---
Cell 4 executed: OPTIMIZED ModelTuner class is now defined.


In [ ]:
# Cell 5: Advanced Tuning with Hyperband

print("--- Cell 5: Initializing and Running the Hyperparameter Tuning ---")

# 1. Instantiate the tuner with our final data and the XGBoost model class
tuner = ModelTuner(
    model_class=xgb.XGBClassifier,
    X=X_final,
    y=y,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE
)

# 2. Run the tuning process for the number of trials we defined in Cell 1
# This will take some time. Optuna will display a progress bar.
print(f"\nStarting XGBoost tuning for {N_TRIALS_XGB} trials with Hyperband pruner...")

# The 'study' object will contain all information about the tuning run
study = tuner.tune(n_trials=N_TRIALS_XGB)

# --- Final Cleanup ---
# We can now delete the final dataframes as they are stored inside the tuner object
del X_final, y
gc.collect()

print("\n--- Cell 5 Complete: Tuning finished. ---")

[I 2025-11-19 14:42:37,503] A new study created in memory with name: no-name-e35bcc8b-d6d4-40bd-afaf-59b86f75daa4


--- Cell 5: Initializing and Running the Hyperparameter Tuning ---
✅ GPU detected. Enabling GPU acceleration.

Starting XGBoost tuning for 50 trials with Hyperband pruner...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-11-19 14:43:10,152] Trial 0 finished with value: 0.920178627023558 and parameters: {'learning_rate': 0.06235186887897628, 'max_depth': 8, 'subsample': 0.8319639926261632, 'colsample_bytree': 0.7167513105012344, 'gamma': 0.33447686201983284, 'lambda': 0.0073677641941099475, 'alpha': 1.934878002581474e-05, 'min_child_weight': 1}. Best is trial 0 with value: 0.920178627023558.
[I 2025-11-19 14:43:11,902] Trial 1 pruned. Trial was pruned at iteration 1.
[I 2025-11-19 14:43:13,702] Trial 2 pruned. Trial was pruned at iteration 1.
[I 2025-11-19 14:43:21,158] Trial 3 pruned. Trial was pruned at iteration 243.
[I 2025-11-19 14:43:22,982] Trial 4 pruned. Trial was pruned at iteration 1.
[I 2025-11-19 14:43:24,861] Trial 5 pruned. Trial was pruned at iteration 1.
[I 2025-11-19 14:44:40,319] Trial 6 finished with value: 0.9183410219859047 and parameters: {'learning_rate': 0.010022354455323469, 'max_depth': 4, 'subsample': 0.6705215842040149, 'colsample_bytree': 0.7517974781579446, 'gamma'

In [ ]:
# Cell 6: Save Best Parameters

print("--- Cell 6: Saving the Best Hyperparameters ---")

# The best parameters are stored in the 'best_params_' attribute of our tuner object
best_params = tuner.best_params_

# Add the GPU parameters back in, as they are not part of the Optuna study
# but are essential for prediction.
best_params.update(tuner.gpu_params)

# Also add n_estimators and the objective, which are fixed but needed for re-instantiation
best_params['n_estimators'] = 2000 # This was the max value we used with early stopping
best_params['objective'] = 'binary:logistic'
best_params['eval_metric'] = 'auc'


# Define the filename for our parameters file
params_filename = "best_params_xgb.json"

# Write the dictionary to a JSON file
with open(params_filename, 'w') as f:
    json.dump(best_params, f, indent=4)

print(f"✅ Best parameters saved to '{params_filename}'")
print("\nFinal parameters to be used for prediction:")
# Print the final, complete dictionary
print(json.dumps(best_params, indent=4))

# --- Final Cleanup ---
# We no longer need the tuner object or the study object
del tuner, study
gc.collect()

print("\n--- Cell 6 Complete ---")

--- Cell 6: Saving the Best Hyperparameters ---
✅ Best parameters saved to 'best_params_xgb.json'

Final parameters to be used for prediction:
{
    "learning_rate": 0.04405373197459871,
    "max_depth": 4,
    "subsample": 0.7775643982264042,
    "colsample_bytree": 0.6562609356380992,
    "gamma": 0.08544972109096269,
    "lambda": 0.036573797057766096,
    "alpha": 0.11168550378002871,
    "min_child_weight": 3,
    "tree_method": "gpu_hist",
    "predictor": "gpu_predictor",
    "n_estimators": 2000,
    "objective": "binary:logistic",
    "eval_metric": "auc"
}

--- Cell 6 Complete ---


In [ ]:
# Cell 7: Predict with Best Parameters (Corrected v3)

print("--- Cell 7 (v3): Generating Predictions with Robust CV Loop ---")

# --- 1. Load the Best Parameters ---
params_filename = "best_params_xgb.json"
print(f"Loading best parameters from '{params_filename}'...")
with open(params_filename, 'r') as f:
    best_params = json.load(f)
print("Parameters loaded successfully.")

# --- 2. Re-create Final Datasets (CORRECTED BLOCK) ---
print("\nRe-creating final datasets for prediction...")
# Load BOTH train and test data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH) # <-- THE MISSING LINE

# Separate the test ID correctly
test_id = test_df['id']
test_df = test_df.drop(columns=['id'])

# Now the rest of the code will work
X = train_df.drop(columns=[TARGET])
y = train_df[TARGET]
X_processed = X.copy()
X_test_processed = test_df.copy()
for col in categorical_features:
    le = label_encoders[col]
    X_processed[col] = le.transform(X_processed[col].astype(str))
    X_test_processed[col] = le.transform(X_test_processed[col].astype(str))
low_cardinality_cats = [col for col in categorical_features if col not in binary_features]
X_processed = pd.get_dummies(X_processed, columns=low_cardinality_cats, drop_first=True)
X_test_processed = pd.get_dummies(X_test_processed, columns=low_cardinality_cats, drop_first=True)
X_processed, X_test_processed = X_processed.align(X_test_processed, join='inner', axis=1)
selector = SelectKBest(score_func=f_classif, k='all')
X_final = pd.DataFrame(selector.fit_transform(X_processed, y), columns=X_processed.columns[selector.get_support()])
X_test_final = pd.DataFrame(selector.transform(X_test_processed), columns=X_processed.columns[selector.get_support()])
print("Final datasets re-created successfully.")

# --- 3. Generate OOF Predictions with a Manual Loop ---
print("\nGenerating Out-of-Fold (OOF) predictions with a manual loop...")
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_preds = np.zeros(len(X_final))
X_final_np = X_final.values
y_np = y.values
for fold, (train_idx, val_idx) in enumerate(tqdm(cv.split(X_final_np, y_np), total=N_SPLITS, desc="OOF Loop")):
    X_train, X_val = X_final_np[train_idx], X_final_np[val_idx]
    y_train, y_val = y_np[train_idx], y_np[val_idx]
    model = xgb.XGBClassifier(**best_params)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[xgb.callback.EarlyStopping(rounds=50, save_best=True)],
              verbose=False)
    preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = preds
oof_score = roc_auc_score(y, oof_preds)
print(f"✅ OOF ROC-AUC Score (Manual Loop): {oof_score:.6f}")

# --- 4. Generate Test Predictions ---
print("\nGenerating Test predictions...")
final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_final, y)
test_preds = final_model.predict_proba(X_test_final)[:, 1]
print("Test predictions generated.")

# --- 5. Final Cleanup ---
del X, y, X_processed, X_test_processed, X_final, X_test_final, train_df, test_df
gc.collect()

print("\n--- Cell 7 Complete ---")


--- Cell 7 (v3): Generating Predictions with Robust CV Loop ---
Loading best parameters from 'best_params_xgb.json'...
Parameters loaded successfully.

Re-creating final datasets for prediction...
Final datasets re-created successfully.

Generating Out-of-Fold (OOF) predictions with a manual loop...


OOF Loop:   0%|          | 0/5 [00:00<?, ?it/s]

✅ OOF ROC-AUC Score (Manual Loop): 0.921482

Generating Test predictions...
Test predictions generated.

--- Cell 7 Complete ---


In [ ]:
# Cell 8: Save Artifacts for Submission & Ensembling (Corrected v2)

print("--- Cell 8: Saving Final Prediction Artifacts ---")

# --- 1. Save OOF Predictions ---
# The OOF predictions are already in the correct order of the original training data.
# The index serves as the implicit ID. We only need the predictions themselves for ensembling.
oof_df = pd.DataFrame({
    'oof_pred_xgb': oof_preds
})
oof_filename = "oof_preds_xgb.csv"
oof_df.to_csv(oof_filename, index=False)
print(f"✅ Out-of-Fold predictions saved to '{oof_filename}'")
display(oof_df.head())


# --- 2. Save Test Predictions ---
# For the test predictions, we DO need the ID to match them correctly.
test_pred_df = pd.DataFrame({
    'id': test_id, # The test IDs we correctly separated in Cell 2
    'test_pred_xgb': test_preds
})
test_pred_filename = "test_preds_xgb.csv"
test_pred_df.to_csv(test_pred_filename, index=False)
print(f"\n✅ Test predictions saved to '{test_pred_filename}'")
display(test_pred_df.head())


# --- 3. Create and Save Submission File ---
# This file requires the 'id' and the final prediction.
submission_df = pd.DataFrame({
    'id': test_id,
    'loan_paid': test_preds # The competition expects the target column to be named 'loan_paid'
})
submission_filename = "submission_xgb.csv"
submission_df.to_csv(submission_filename, index=False)
print(f"\n✅ Submission file created: '{submission_filename}'")
display(submission_df.head())

print("\n--- Cell 8 Complete: All artifacts saved. Notebook finished! ---")


--- Cell 8: Saving Final Prediction Artifacts ---
✅ Out-of-Fold predictions saved to 'oof_preds_xgb.csv'


,oof_pred_xgb
0,0.982214
1,0.578179
2,0.932957
3,0.890246
4,0.975491



✅ Test predictions saved to 'test_preds_xgb.csv'


,id,test_pred_xgb
0,593994,0.946152
1,593995,0.982046
2,593996,0.425929
3,593997,0.919987
4,593998,0.971120



✅ Submission file created: 'submission_xgb.csv'


,id,loan_paid
0,593994,0.946152
1,593995,0.982046
2,593996,0.425929
3,593997,0.919987
4,593998,0.971120



--- Cell 8 Complete: All artifacts saved. Notebook finished! ---
